In [1]:
from ragaai_catalyst import trace_llm, trace_tool
from dotenv import load_dotenv
load_dotenv()

from ragaai_catalyst.tracers import Tracer
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_llm, trace_tool
import os

catalyst = RagaAICatalyst(
    access_key=os.getenv("RAGAAI_CATALYST_ACCESS_KEY"),
    secret_key=os.getenv("RAGAAI_CATALYST_SECRET_KEY"),
    base_url=os.getenv("RAGAAI_CATALYST_BASE_URL"),
)

project = catalyst.create_project(
    project_name="agent_query_planner",
    usecase="Agentic Application"
)
# Initialize tracer
tracer = Tracer(
    project_name="agent_query_planner",
    dataset_name="file",
    tracer_type="Agentic",
)

init_tracing(catalyst=catalyst, tracer=tracer)

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.llms.openai import OpenAI

llm = OpenAI(temperature=0, model="gpt-4")

# Download and prepare data
import os
import requests

# Create directory
os.makedirs('data/10q', exist_ok=True)

# URLs
urls = [
    ('uber_10q_march_2022.pdf', 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10q/uber_10q_march_2022.pdf'),
    ('uber_10q_june_2022.pdf', 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10q/uber_10q_june_2022.pdf'),
    ('uber_10q_sept_2022.pdf', 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10q/uber_10q_sept_2022.pdf')
]

# Download files
for filename, url in urls:
    response = requests.get(url)
    with open(f'data/10q/{filename}', 'wb') as f:
        f.write(response.content)

# Load and index documents
march_2022 = SimpleDirectoryReader(
    input_files=["./data/10q/uber_10q_march_2022.pdf"]
).load_data()
june_2022 = SimpleDirectoryReader(
    input_files=["./data/10q/uber_10q_june_2022.pdf"]
).load_data()
sept_2022 = SimpleDirectoryReader(
    input_files=["./data/10q/uber_10q_sept_2022.pdf"]
).load_data()

march_index = VectorStoreIndex.from_documents(march_2022)
june_index = VectorStoreIndex.from_documents(june_2022)
sept_index = VectorStoreIndex.from_documents(sept_2022)

march_engine = march_index.as_query_engine(similarity_top_k=3, llm=llm)
june_engine = june_index.as_query_engine(similarity_top_k=3, llm=llm)
sept_engine = sept_index.as_query_engine(similarity_top_k=3, llm=llm)

from llama_index.core.tools import QueryEngineTool

query_tool_sept = QueryEngineTool.from_defaults(
    query_engine=sept_engine,
    name="sept_2022",
    description=(
        f"Provides information about Uber quarterly financials ending"
        f" September 2022"
    ),
)
query_tool_june = QueryEngineTool.from_defaults(
    query_engine=june_engine,
    name="june_2022",
    description=(
        f"Provides information about Uber quarterly financials ending June"
        f" 2022"
    ),
)
query_tool_march = QueryEngineTool.from_defaults(
    query_engine=march_engine,
    name="march_2022",
    description=(
        f"Provides information about Uber quarterly financials ending March"
        f" 2022"
    ),
)

from llama_index.core.tools import QueryPlanTool
from llama_index.core import get_response_synthesizer

response_synthesizer = get_response_synthesizer()
query_plan_tool = QueryPlanTool.from_defaults(
    query_engine_tools=[query_tool_sept, query_tool_june, query_tool_march],
    response_synthesizer=response_synthesizer,
)

from llama_index.agent.openai import OpenAIAgent
from llama_index.llms.openai import OpenAI

agent = OpenAIAgent.from_tools(
    [query_plan_tool],
    max_function_calls=10,
    llm=OpenAI(temperature=0, model="gpt-4-0613"),
    verbose=True,
)

# Define traced functions for different query types
@trace_llm("analyze_risk_factors")
async def get_risk_factors(agent):
    """Analyze risk factors for September 2022"""
    response = agent.query("What were the risk factors in sept 2022?")
    return response

@trace_llm("analyze_revenue")
async def analyze_revenue(agent):
    """Analyze revenue growth across quarters"""
    response = agent.query(
        "Analyze Uber revenue growth in March, June, and September"
    )
    return response

@trace_llm("analyze_risk_changes")
async def analyze_risk_changes(agent):
    """Analyze changes in risk factors across quarters"""
    response = agent.query(
        "Analyze changes in risk factors in march, june, and september for Uber"
    )
    return response

@trace_tool("compare_documents")
async def compare_documents(agent):
    """Compare revenue growth and risk factors between March and September"""
    response = agent.query(
        "First look at Uber's revenue growth and risk factors in March, "
        + "then revenue growth and risk factors in September, and then compare and"
        " contrast the two documents?"
    )
    return response

# Execute queries with tracing
with tracer:
    # Get risk factors for September 2022
    response = await get_risk_factors(agent)
    print("\nRisk Factors (Sept 2022):")
    print(str(response))

    # Analyze revenue across quarters
    response = await analyze_revenue(agent)
    print("\nRevenue Analysis:")
    print(str(response))

    # Analyze risk factor changes
    response = await analyze_risk_changes(agent)
    print("\nRisk Factor Changes:")
    print(str(response))

    # Compare documents
    response = await compare_documents(agent)
    print("\nDocument Comparison:")
    print(str(response))

INFO:httpx:HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"


Token(s) set successfully
Project Created Successfully with name agent_query_planner & usecase Agentic Application


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Added user message to memory: What were the risk factors in sept 2022?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Executed query, got response.
Query: What is Uber's revenue growth and risk factors in September 2022?
Response: Uber's revenue in September 2022 increased by 72% for the three months ended compared to the same period in 2021, and by 99% for the nine months ended compared to the same period in 2021. This growth was primarily due to an increase in Gross Bookings, recovery from the impacts of COVID-19, and a significant increase in Freight Gross Bookings resulting from the acquisition of Transplace in the fourth quarter of 2021.

However, Uber also faces several risk factors. These include the ability to attract and retain new platform users, increased competition, the ability to expand operations in new and existing markets, maintaining an adequate growth rate, keeping pace with technological changes, changes in regulations affecting the business, harm to the brand or reputation, and the duration and severity of the COVID-19 pandemic. Additionally, a significant portion of Uber's expens

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Executed query, got response.
Query: Compare and contrast Uber's revenue growth and risk factors in March 2022 and September 2022.
Response: In March 2022, Uber's revenue grew by 136% year-over-year, driven by specific factors such as growth in the Freight business and accounting model changes. On the other hand, in September 2022, the revenue growth was 72% for the three months ended and 99% for the nine months ended compared to the same period in 2021. This growth was attributed to increased Gross Bookings and recovery from the impacts of COVID-19, including the acquisition of Transplace.

Regarding risk factors, in March 2022, Uber faced uncertainties due to the adverse impact of the COVID-19 pandemic on its business, particularly in reducing global demand for Mobility offerings. In September 2022, the risk factors included challenges related to attracting and retaining platform users, increased competition, regulatory changes, technological advancements, and the ongoing impact of t

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ragaai_catalyst.tracers.agentic_tracing.utils.zip_list_of_unique_files: Zip file created successfully.
INFO:ragaai_catalyst.tracers.agentic_tracing.tracers.base: Traces saved successfully.



Document Comparison:
In March 2022, Uber's revenue grew by 136% year-over-year, driven by specific factors such as growth in the Freight business and accounting model changes. The risk factors during this period were uncertainties due to the adverse impact of the COVID-19 pandemic on its business, particularly in reducing global demand for Mobility offerings.

On the other hand, in September 2022, the revenue growth was 72% for the three months ended and 99% for the nine months ended compared to the same period in 2021. This growth was attributed to increased Gross Bookings and recovery from the impacts of COVID-19, including the acquisition of Transplace. The risk factors during this period included challenges related to attracting and retaining platform users, increased competition, regulatory changes, technological advancements, and the ongoing impact of the pandemic. Additionally, Uber's fixed expenses and investments posed a risk if revenue fell below expectations, potentially le